# Sprint 4: Prompt Augmentation and Response Generation

This notebook demonstrates the full RAG response flow:
1. Query retrieval
2. Prompt augmentation
3. LLM response generation
4. Dataset-based evaluation

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

PORT = 8017
BASE_URL = f"http://127.0.0.1:{PORT}"

def post_json(path: str, payload: dict, timeout: int = 120):
    req = urllib.request.Request(
        f"{BASE_URL}{path}",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.status, json.loads(resp.read().decode("utf-8"))

def get_json(path: str, timeout: int = 60):
    with urllib.request.urlopen(f"{BASE_URL}{path}", timeout=timeout) as resp:
        return resp.status, json.loads(resp.read().decode("utf-8"))


## Authenticate and Capture Token

In [ ]:
from ailab.utils.azure import get_ailab_bearer_token, set_env_bearer_token

auth_result = get_ailab_bearer_token()
set_env_bearer_token(auth_result.token)
print("Authenticated. Token expires on:", auth_result.expires_on)


## Start FastAPI Server

In [ ]:
ROOT = Path.cwd()
SRC_DIR = ROOT / "src"

cmd = [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", str(PORT)]
server_env = dict(os.environ)
server_env["PYTHONPATH"] = str(SRC_DIR)

server_proc = subprocess.Popen(cmd, cwd=str(ROOT), env=server_env)

ready = False
for _ in range(40):
    try:
        status_code, _ = get_json("/health", timeout=2)
        if status_code == 200:
            ready = True
            break
    except Exception:
        time.sleep(0.5)

print("Server ready:", ready)


## Ingest Dataset (Small Batch)

In [ ]:
status_code, ingest_result = post_json(
    "/vectordb/ingest",
    {
        "source": "hf://datasets/rag-datasets/rag-mini-wikipedia/data/passages.parquet/part.0.parquet",
        "limit": 8,
    },
    timeout=300,
)
print("Status:", status_code)
print(json.dumps(ingest_result, indent=2)[:1000])


## Run End-to-End RAG Query

In [ ]:
query_payload = {"query": "What is artificial intelligence?", "top_k": 3}
status_code, rag_result = post_json("/rag/query", query_payload, timeout=180)
print("Status:", status_code)
print("Answer:", rag_result.get("answer"))
print("\nPrompt preview:")
print(rag_result.get("prompt", "")[:800])
print("\nRetrieved docs:", len(rag_result.get("results", [])))


## Evaluate Against QA Dataset

In [ ]:
eval_payload = {
    "source": "hf://datasets/rag-datasets/rag-mini-wikipedia/data/test.parquet/part.0.parquet",
    "limit": 5,
    "top_k": 3,
}
status_code, eval_result = post_json("/rag/evaluate", eval_payload, timeout=600)
print("Status:", status_code)
print("Summary:", json.dumps(eval_result.get("summary", {}), indent=2))
print("First item:", json.dumps(eval_result.get("items", [])[0], indent=2) if eval_result.get("items") else "No items")


## Verify Relevant Endpoints

In [ ]:
checks = [
    ("GET", "/health"),
    ("GET", "/vectordb/status"),
    ("POST", "/vectordb/query"),
    ("POST", "/rag/query"),
    ("POST", "/rag/evaluate"),
]

for method, endpoint in checks:
    print(f"{method} {endpoint}")


## Cleanup

In [ ]:
if "server_proc" in globals() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except Exception:
        server_proc.kill()
print("Server stopped.")


## Summary

Sprint 4 observability complete:
- Augmented prompt is generated from retrieved context
- GPT answer is returned by `/rag/query`
- Dataset quality metrics are produced by `/rag/evaluate`